<a href="https://colab.research.google.com/github/Prktriesitout/Langchain_Projects/blob/main/Project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

A university research office and multiple engineering teams routinely ingest dozens of new research papers each week. Researchers and product managers need fast, reliable summaries that extract the core problem, the method proposed, and the key findings so they can triage, synthesize literature reviews, and identify promising approaches quickly. Manually reading every paper is time-consuming and creates a bottleneck for decision-making.

An automated AI Research Assistant that reads uploaded PDF papers and produces structured three-part summaries (Problem / Method / Findings) will accelerate literature review, improve knowledge discovery, and reduce time-to-insight for researchers.

Build an automated Contract/Research Paper Summarizer that:

Accepts a PDF (example filename: default.filename).

Download the research paper from here >> research_paper.pdf

Extracts readable text from the PDF and processes the content.

Uses a prompt-driven LLM workflow to produce a structured summary with three sections:

Problem Statement

Proposed Method

Key Findings

Returns the summary in a predictable, machine-friendly format suitable for ingestion into a knowledge base.

Works without using proprietary OpenAI APIs (use open-source LLMs).

In [2]:
!pip install langchain langchain-community langchain-core langchain-huggingface pypdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 62.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.6 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [langchain-community]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [31]:
!pip install -U langchain-cohere

  Attempting uninstall: cohere
    Found existing installation: cohere 7.3.1
    Uninstalling cohere-7.3.1:
      Successfully uninstalled cohere-7.3.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [langchain-cohere]


In [32]:
from google.colab import userdata
from langchain_cohere import ChatCohere
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load and split the PDF
pdf_file = "research_paper.pdf"
chunk_size = 1000
chunk_overlap = 100

print(f"Loading {pdf_file}...")
loader = PyPDFLoader(pdf_file)
pages = loader.load()
splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, chunk_overlap=chunk_overlap
)
docs = splitter.split_documents(pages)

# 2. Initialize Cohere LLM for LangChain
cohere_api_key = userdata.get("cohere")

llm = ChatCohere(
    cohere_api_key=cohere_api_key,
    model="command-a-plus-05-2026",
    max_tokens=512,
    temperature=0.6,
)

# 3. Create the LangChain prompt and chain
template = """
You are an expert assistant for summarizing academic and legal documents.
### Document Chunk:
{document_chunk}

### Instructions:
Extract and summarize the following sections:
1. Problem Statement
2. Proposed Method
3. Key Findings

Return the summary in JSON format with keys:
"problem_statement", "proposed_method", and "key_findings".
"""

prompt = PromptTemplate.from_template(template)
chain = prompt | llm | StrOutputParser()

# 4. Run summarization on the first 3 chunks
print("Summarizing document...")
for i, doc in enumerate(docs[:3]):
    summary = chain.invoke({"document_chunk": doc.page_content})
    print(f"Summary for chunk {i+1}:\n{summary}\n")


Loading research_paper.pdf...
Summarizing document...
Summary for chunk 1:
{
  "problem_statement": "Current AI agent benchmarks focus narrowly on accuracy, ignoring cost and other metrics. This leads to overly complex, expensive agents and mistaken conclusions about what drives accuracy gains. Additionally, the needs of model developers and downstream application developers are conflated in benchmarking, making evaluation less useful for real‑world use.",
  "proposed_method": "Introduce a new evaluation goal that jointly optimizes accuracy and cost. The authors design and implement a concrete optimization approach that balances these two metrics, demonstrating its ability to significantly cut costs while preserving high accuracy.",
  "key_findings": "The joint optimization reduces agent cost dramatically without sacrificing accuracy, showing the practical benefit of cost‑aware evaluation. The analysis also reveals that existing benchmarks misattribute accuracy improvements and that se

In [33]:
!pip install -q gradio langchain-cohere langchain-community langchain-text-splitters pypdf

In [34]:
import json
import re

import gradio as gr
from google.colab import userdata
from langchain_cohere import ChatCohere
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ---------- LLM ----------
llm = ChatCohere(
    cohere_api_key=userdata.get("cohere"),
    model="command-a-plus-05-2026",
    max_tokens=1024,
    temperature=0.3,
)

# ---------- Prompts ----------
chunk_prompt = PromptTemplate.from_template("""
You are an expert assistant for summarizing academic and legal documents.

### Document Chunk:
{text}

### Instructions:
Write concise notes on any of the following that appear in this chunk:
problem statement, proposed method, key findings.
If something is not present, skip it.
""")

final_prompt = PromptTemplate.from_template("""
You are an expert assistant for summarizing academic and legal documents.

### Notes from the document:
{notes}

### Instructions:
Combine the notes into one coherent summary.
Return ONLY valid JSON with these keys:
"problem_statement", "proposed_method", "key_findings".
""")

chunk_chain = chunk_prompt | llm | StrOutputParser()
final_chain = final_prompt | llm | StrOutputParser()


# ---------- Helpers ----------
def parse_json(text: str) -> dict:
    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
    return json.loads(text)


def format_markdown(data: dict) -> str:
    def fmt(v):
        return "\n".join(f"- {i}" for i in v) if isinstance(v, list) else str(v)

    return (
        f"### Problem Statement\n{fmt(data.get('problem_statement', 'N/A'))}\n\n"
        f"### Proposed Method\n{fmt(data.get('proposed_method', 'N/A'))}\n\n"
        f"### Key Findings\n{fmt(data.get('key_findings', 'N/A'))}"
    )


# ---------- Main function ----------
def summarize_pdf(pdf_path, max_chunks, progress=gr.Progress()):
    if pdf_path is None:
        return "⚠️ Please upload a PDF file.", None

    try:
        progress(0.05, desc="Reading PDF...")
        pages = PyPDFLoader(pdf_path).load()
        splitter = RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=200)
        docs = splitter.split_documents(pages)[: int(max_chunks)]

        if not docs:
            return "⚠️ No text could be extracted (is the PDF scanned?).", None

        notes = []
        for i, doc in enumerate(docs):
            progress((i + 1) / (len(docs) + 1), desc=f"Summarizing chunk {i+1}/{len(docs)}...")
            notes.append(chunk_chain.invoke({"text": doc.page_content}))

        progress(0.95, desc="Combining summary...")
        raw = final_chain.invoke({"notes": "\n\n".join(notes)})

        try:
            data = parse_json(raw)
            return format_markdown(data), data
        except json.JSONDecodeError:
            return raw, None

    except Exception as e:
        return f"❌ Error: {e}", None


# ---------- UI ----------
with gr.Blocks(theme=gr.themes.Soft(), title="PDF Summarizer") as demo:
    gr.Markdown(
        "# 📄 PDF Summarizer\n"
        "Upload a research or legal PDF to get its problem statement, method, and key findings."
    )

    with gr.Row():
        with gr.Column(scale=1):
            pdf_input = gr.File(label="Upload PDF", file_types=[".pdf"], type="filepath")
            max_chunks = gr.Slider(
                3, 30, value=10, step=1,
                label="Max chunks to process",
                info="Higher = more coverage, slower.",
            )
            run_btn = gr.Button("Summarize", variant="primary")
            clear_btn = gr.ClearButton()

        with gr.Column(scale=2):
            summary_md = gr.Markdown(label="Summary")
            with gr.Accordion("Raw JSON", open=False):
                summary_json = gr.JSON()

    run_btn.click(summarize_pdf, [pdf_input, max_chunks], [summary_md, summary_json])
    clear_btn.add([pdf_input, summary_md, summary_json])

demo.launch(debug=True)

/tmp/ipykernel_552/1447875112.py:99: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="PDF Summarizer") as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f608f7b88da871aaa0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://f608f7b88da871aaa0.gradio.live


In [35]:
import json
import re
import time

import gradio as gr
from google.colab import userdata
from langchain_cohere import ChatCohere
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

API_KEY = userdata.get("cohere")
MODEL = "command-a-plus-05-2026"

# Short, fast model calls for intermediate notes
notes_llm = ChatCohere(cohere_api_key=API_KEY, model=MODEL, max_tokens=300, temperature=0.2)
# Final merge call needs a bit more room
final_llm = ChatCohere(cohere_api_key=API_KEY, model=MODEL, max_tokens=700, temperature=0.3)

chunk_prompt = PromptTemplate.from_template("""
Summarize this excerpt from a document in at most 5 short bullet points.
Focus only on: problem statement, proposed method, key findings. Skip anything else.

{text}
""")

final_prompt = PromptTemplate.from_template("""
You are an expert assistant for summarizing academic and legal documents.

### Notes:
{notes}

### Instructions:
Combine the notes into one coherent summary.
Return ONLY valid JSON with keys:
"problem_statement", "proposed_method", "key_findings".
""")

chunk_chain = chunk_prompt | notes_llm | StrOutputParser()
final_chain = final_prompt | final_llm | StrOutputParser()


def parse_json(text):
    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
    return json.loads(text)


def format_markdown(data):
    def fmt(v):
        return "\n".join(f"- {i}" for i in v) if isinstance(v, list) else str(v)
    return (
        f"### Problem Statement\n{fmt(data.get('problem_statement', 'N/A'))}\n\n"
        f"### Proposed Method\n{fmt(data.get('proposed_method', 'N/A'))}\n\n"
        f"### Key Findings\n{fmt(data.get('key_findings', 'N/A'))}"
    )


def pick_evenly(items, n):
    """Pick n items spread evenly across the list (covers whole document)."""
    if len(items) <= n:
        return items
    step = len(items) / n
    return [items[int(i * step)] for i in range(n)]


def summarize_pdf(pdf_path, max_chunks, progress=gr.Progress()):
    if pdf_path is None:
        return "⚠️ Please upload a PDF file.", None

    try:
        start = time.time()
        progress(0.1, desc="Reading PDF...")
        pages = PyPDFLoader(pdf_path).load()
        splitter = RecursiveCharacterTextSplitter(chunk_size=6000, chunk_overlap=200)
        docs = pick_evenly(splitter.split_documents(pages), int(max_chunks))

        if not docs:
            return "⚠️ No text could be extracted (is the PDF scanned?).", None

        progress(0.3, desc=f"Summarizing {len(docs)} chunks in parallel...")
        notes = chunk_chain.batch(
            [{"text": d.page_content} for d in docs],
            config={"max_concurrency": 5},
        )

        progress(0.85, desc="Combining summary...")
        raw = final_chain.invoke({"notes": "\n\n".join(notes)})

        elapsed = time.time() - start
        try:
            data = parse_json(raw)
            return format_markdown(data) + f"\n\n---\n*Done in {elapsed:.1f}s*", data
        except json.JSONDecodeError:
            return raw, None

    except Exception as e:
        return f"❌ Error: {e}", None


with gr.Blocks(theme=gr.themes.Soft(), title="PDF Summarizer") as demo:
    gr.Markdown(
        "# 📄 PDF Summarizer\n"
        "Upload a research or legal PDF to get its problem statement, method, and key findings."
    )
    with gr.Row():
        with gr.Column(scale=1):
            pdf_input = gr.File(label="Upload PDF", file_types=[".pdf"], type="filepath")
            max_chunks = gr.Slider(
                2, 15, value=5, step=1,
                label="Chunks to process",
                info="Lower = faster. Chunks are spread evenly across the PDF.",
            )
            run_btn = gr.Button("Summarize", variant="primary")
            clear_btn = gr.ClearButton()
        with gr.Column(scale=2):
            summary_md = gr.Markdown()
            with gr.Accordion("Raw JSON", open=False):
                summary_json = gr.JSON()

    run_btn.click(summarize_pdf, [pdf_input, max_chunks], [summary_md, summary_json])
    clear_btn.add([pdf_input, summary_md, summary_json])

demo.launch(debug=True)

/tmp/ipykernel_552/3074712625.py:101: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="PDF Summarizer") as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://32b46a0221e013c072.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://32b46a0221e013c072.gradio.live
